# Regression Analysis and the Simple Linear Model

**STA 506 2.0 Linear Regression Analysis** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta506.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Purpose and workflow](#theory-01)
- [Why plots precede summaries](#theory-02)
- [Model and assumptions](#theory-03)
- [Deriving the fitted coefficients](#theory-04)
- [Variances and parameter intervals](#theory-05)
- [Regression ANOVA and R²](#theory-06)
- [Mean response and prediction](#theory-07)

## Detailed theory · Simple linear regression from model to inference

<a id="theory-01"></a>

### Purpose and workflow
Regression models a conditional response in relation to predictors for description, estimation, prediction or control. Correlation describes association without a response/predictor role. State the problem, select variables, collect data, specify the model, fit, validate and interpret. Historical, observational and randomized experimental data support different causal conclusions. Galton's regression-to-the-mean history motivates the name but is not a rule that individuals must move toward a population average.

<a id="theory-02"></a>

### Why plots precede summaries
Different datasets can share means, SDs, correlations and a fitted line yet have very different shapes, as illustrated by the Anscombe-quartet idea. Plot observations, look for curvature and influential cases, and inspect the predictor range. A high R² cannot repair a wrong mean structure.

<a id="theory-03"></a>

### Model and assumptions
$$Y_i=\beta_0+\beta_1x_i+\epsilon_i,\quad E(Y_i\mid x_i)=\beta_0+\beta_1x_i.$$
Linearity refers to the coefficients: a model can include known transforms of x and remain linear in its parameters. For unbiased least squares use zero conditional error mean and a full-rank design. Common variance and uncorrelated errors support the Gauss–Markov minimum-variance linear-unbiased result. Independent normal errors add exact small-sample t/F inference; they are not required merely to minimize squared residuals.

<a id="theory-04"></a>

### Deriving the fitted coefficients
Minimize $S=\sum(y_i-b_0-b_1x_i)^2$. Setting derivatives to zero gives $\sum e_i=0$ and $\sum x_ie_i=0$. Define $S_{xx}=\sum(x_i-\bar x)^2$, $S_{xy}=\sum(x_i-\bar x)(y_i-\bar y)$:
$$\hat\beta_1=S_{xy}/S_{xx},\quad\hat\beta_0=\bar y-\hat\beta_1\bar x.$$
Sxx must be positive. The fitted line passes through the sample means, and residuals are orthogonal to fitted values under the intercept model.

<a id="theory-05"></a>

### Variances and parameter intervals
$$s^2=MSE=SSE/(n-2),\quad SE(\hat\beta_1)=s/\sqrt{S_{xx}},$$
$$SE(\hat\beta_0)=s\sqrt{1/n+\bar x^2/S_{xx}}.$$
Test a coefficient using $(\hat\beta_j-\beta_{j0})/SE(\hat\beta_j)$ against t with n−2 df under the normal-error model. A two-sided interval is estimate ± the matching t quantile times SE. The normal-likelihood ML residual variance uses SSE/n and is biased at finite n; it differs from the unbiased MSE.

<a id="theory-06"></a>

### Regression ANOVA and R²
With an intercept, $SST=SSR+SSE$, df are n−1,1,n−2, and $F=(SSR/1)/MSE$ tests zero slope. For this one-predictor two-sided comparison, F=t². $R^2=1-SSE/SST$ is the sample variation fraction accounted for by the fitted model; in SLR it equals r². It depends on the predictor range and is not a causal measure.

<a id="theory-07"></a>

### Mean response and prediction
At x0, $\hat y_0=\hat\beta_0+\hat\beta_1x_0$ has mean-response SE $s\sqrt{1/n+(x_0-\bar x)^2/S_{xx}}$. A new-observation prediction SE is $s\sqrt{1+1/n+(x_0-\bar x)^2/S_{xx}}$. The extra 1 represents new noise. Use a square root of variance, not variance, as the interval multiplier. Both intervals widen away from the data centre; extrapolation can fail regardless of a narrow calculated interval.

## Concepts covered

**Course points:** Simple regression; least squares; slope/intercept; residuals; R-squared; coefficient uncertainty.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Simple regression: sums, fitted values and residuals

### Question and assumptions

Fit y on x for x=[1,2,3,4] and y=[2,3,5,4], with an intercept.

### Procedure

1. Calculate centred cross-products and squared x deviations.
2. Use slope=Sxy/Sxx and intercept=mean(y)−slope×mean(x).
3. Calculate fitted values, residuals and SSE. Verify that residuals sum to zero.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| x | y | x−mean x | y−mean y | Squared x deviation | Cross-product |
| --- | --- | --- | --- | --- | --- |
| 1 | 2 | -1.5 | -1.5 | 2.25 | 2.25 |
| 2 | 3 | -0.5 | -0.5 | 0.25 | 0.25 |
| 3 | 5 | 0.5 | 1.5 | 0.25 | 0.75 |
| 4 | 4 | 1.5 | 0.5 | 2.25 | 0.75 |

| x | Observed y | Fitted y | Residual | Squared residual |
| --- | --- | --- | --- | --- |
| 1 | 2 | 2.3 | -0.3 | 0.09 |
| 2 | 3 | 3.1 | -0.1 | 0.01 |
| 3 | 5 | 3.9 | 1.1 | 1.21 |
| 4 | 4 | 4.7 | -0.7 | 0.49 |

| Quantity | Value |
| --- | --- |
| Sxx | 5 |
| Sxy | 4 |
| Slope | 0.8 |
| Intercept | 1.5 |
| SSE | 1.8 |
| R² | 0.64 |
| Error variance SSE/(n−2) | 0.9 |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


x=[1,2,3,4]; y=[2,3,5,4]; xb=mean(x); yb=mean(y)
rows=[[a,b,a-xb,b-yb,(a-xb)**2,(a-xb)*(b-yb)] for a,b in zip(x,y)]
show_table(["x","y","x−mean x","y−mean y","Squared x deviation","Cross-product"],rows)
Sxx=sum(r[4] for r in rows); Sxy=sum(r[5] for r in rows); slope=Sxy/Sxx; intercept=yb-slope*xb
fit=[intercept+slope*a for a in x]; e=[b-f for b,f in zip(y,fit)]
show_table(["x","Observed y","Fitted y","Residual","Squared residual"],[[a,b,f,r,r*r] for a,b,f,r in zip(x,y,fit,e)])
SSE=sum(r*r for r in e); SST=sum((b-yb)**2 for b in y)
show_table(["Quantity","Value"],[["Sxx",Sxx],["Sxy",Sxy],["Slope",slope],["Intercept",intercept],["SSE",SSE],["R²",1-SSE/SST],["Error variance SSE/(n−2)",SSE/(len(x)-2)]])
assert abs(sum(e))<1e-12 and abs(sum(a*r for a,r in zip(x,e)))<1e-12


### Decision and interpretation

The intercept model makes residuals sum to zero and makes them orthogonal to x. These identities are useful checks; they do not establish normality or independence.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
x=rng.uniform(2,12,70); y=8+3.2*x+rng.normal(0,7,70)
slope,intercept,r,p,se=stats.linregress(x,y); fit=intercept+slope*x; e=y-fit
print(f'yhat={intercept:.2f}+{slope:.2f}x | R2={r*r:.3f} | slope p={p:.4g}')
fig,ax=plt.subplots(1,2); ax[0].scatter(x,y); ax[0].plot(x,fit,color='red'); ax[1].scatter(fit,e); ax[1].axhline(0,color='k',ls='--'); plt.show()

### Practice set · Simple regression
1. Interpret ŷ=12+2.5x when x is study hours and y is score points.
2. If SSE=80 and SST=200, calculate R².
3. What does a fan-shaped residual plot suggest?
4. Why can a significant slope still be noncausal?

<details><summary>Answer sketches</summary>
1. Predicted mean score rises 2.5 points per additional study hour; intercept predicts 12 at zero hours, if meaningful. 2. R²=1−80/200=.60. 3. Nonconstant error variance. 4. Confounding, selection, reverse causation, or observational design can explain the association.
</details>

### Debrief
Name one residual pattern that challenges linearity.

Record the formula, assumptions, and one follow-up question.